# Chain-depth calibration on Colab

Chooses the chain depths `d` and the middle budget for the main grid. The single agent runs on development problems at several depths, with every prompt holding the same number of equations, so only the dependency path changes with `d`. Useful depths keep single-agent accuracy at the middle budget away from 0% and 100%, so the other systems have room to do better or worse.

These problems use the development seed. The main grid must use a different seed, so prompts are never tuned on test problems.

Use **Runtime > Change runtime type > L4 GPU**. Results go to `MyDrive/small-multi-agents/results/calibrate_depth/`, with the generated problems, a `run.log`, and summary CSV files. If the session disconnects, rerun every cell. Runs resume from the last saved batch.

In [1]:
DEPTHS = [1, 2, 4, 6]
EQUATIONS = 14          # per prompt; the distractor chain fills EQUATIONS - 2 * d
PAIRS_PER_DEPTH = 10
DEV_SEED = 3001         # the main grid uses a different seed
BUDGETS = [4000, 8000, 16000]
SAMPLING_SEED = 42

assert all(1 <= 2 * d < EQUATIONS for d in DEPTHS), "each prompt needs at least one distractor update"

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [3]:
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/small-multi-agents/results/calibrate_depth"
!mkdir -p {OUT}

Mounted at /content/drive


In [4]:
REPO = "/content/small-multi-agents"
!test -d {REPO} || git clone https://github.com/liatzheng/small-multi-agents.git {REPO}
%cd {REPO}
!git pull --ff-only && git log -1 --format='%H %s'

Cloning into '/content/small-multi-agents'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 126 (delta 69), reused 83 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 103.78 KiB | 2.31 MiB/s, done.
Resolving deltas: 100% (69/69), done.
/content/small-multi-agents
Already up to date.
d566b190de6aa0560676ea33610a9bb568528417 Created using Colab


vLLM 0.33.0 brings its own matching torch, torchvision, and transformers, replacing Colab's. The runner runs as a subprocess, so no runtime restart is needed.

In [5]:
!pip install -q -r requirements-vllm.txt
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__, torch.cuda.get_device_name())" | tee {OUT}/environment.txt
!git rev-parse HEAD | tee -a {OUT}/environment.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
vllm 0.33.0 torch 2.14.0+cu130 transformers 5.20.2 NVIDIA L4
d566b190de6aa0560676ea33610a9bb5

## Problems

Generation is deterministic, so rerunning this cell rewrites the same file and resuming stays valid.

In [6]:
PROBLEMS = f"{OUT}/problems.jsonl"
!rm -f {PROBLEMS}
for d in DEPTHS:
    distractor = EQUATIONS - 2 * d
    !python generate_problems.py --pairs {PAIRS_PER_DEPTH} --seed {DEV_SEED} --chain-length {d} --distractor-length {distractor} >> {PROBLEMS}
!wc -l {PROBLEMS}

80 /content/drive/MyDrive/small-multi-agents/results/calibrate_depth/problems.jsonl


## Run

The single agent only, at every budget. Everything the runner prints is also appended to `run.log`.

In [7]:
from notebook_tools import run_logged

run_logged(["python", "run_vllm.py", PROBLEMS, "--out-dir", OUT, "--systems", "single_agent",
            "--budgets", *map(str, BUDGETS), "--seed", str(SAMPLING_SEED), "--resume"], f"{OUT}/run.log")

=== python run_vllm.py /content/drive/MyDrive/small-multi-agents/results/calibrate_depth/problems.jsonl --out-dir /content/drive/MyDrive/small-multi-agents/results/calibrate_depth --systems single_agent --budgets 4000 8000 16000 --seed 42 --resume
=== started 2026-11-20T16:42:34+00:00
Resuming after 80 finished problems in /content/drive/MyDrive/small-multi-agents/results/calibrate_depth/b4000/single_agent.jsonl
Resuming after 80 finished problems in /content/drive/MyDrive/small-multi-agents/results/calibrate_depth/b8000/single_agent.jsonl
W1120 16:42:54.105000 4054 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1120 16:42:54.105000 4054 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1120 16:42:54.106000 4054 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=4054) INFO 11-20 16:42:59 [startup.py:96] non-default args: {'

## Results

Accuracy by depth, problem kind, and budget. `length_limit` is the share of problems where the model ran out of tokens before answering.

In [8]:
import json
from pathlib import Path

import pandas as pd

rows = []
for path in sorted(Path(OUT).glob("b*/single_agent.jsonl")):
    for line in path.read_text().splitlines():
        r = json.loads(line)
        rows.append({
            "d": r["chain_length"], "kind": r["kind"], "budget": r["budget"], "pair_id": r["pair_id"],
            "status": r["status"], "correct": r["correct"], "output": r["output_tokens"],
            "total": r["total_tokens"],
        })
df = pd.DataFrame(rows)
df.to_csv(f"{OUT}/summary_problems.csv", index=False)

by_depth = df.groupby(["d", "kind", "budget"]).agg(
    accuracy=("correct", "mean"),
    problems=("correct", "size"),
    length_limit=("status", lambda s: (s == "length_limit").mean()),
    mean_output=("output", "mean"),
).round(2)
by_depth.to_csv(f"{OUT}/summary_by_depth.csv")
by_depth

accuracy  problems  length_limit  mean_output
d kind        budget                                               
1 dependent   4000         0.2        10           0.8       3608.1
              8000         1.0        10           0.0       4620.1
              16000        1.0        10           0.0       4164.6
  independent 4000         0.4        10           0.6       3441.2
              8000         1.0        10           0.0       3520.1
              16000        1.0        10           0.0       3723.2
2 dependent   4000         0.2        10           0.8       3525.0
              8000         1.0        10           0.0       4099.2
              16000        1.0        10           0.0       4439.4
  independent 4000         0.5        10           0.5       3483.7
              8000         1.0        10           0.0       4065.1
              16000        1.0        10           0.0       4001.0
4 dependent   4000         0.1        10           0.9       3629.2
              8000         1.0        10           0.0       4130.8
              16000        1.0        10           0.0       3987.5
  independent 4000         0.5        10           0.5       3501.1
              8000         1.0        10           0.0       3876.9
              16000        1.0        10           0.0       3918.1
6 dependent   4000         0.3        10           0.7       3575.5
              8000         1.0        10           0.0       4163.9
              16000        1.0        10           0.0       4360.0
  independent 4000         0.5        10           0.5       3481.7
              8000         1.0        10           0.0       3725.0
              16000        1.0        10           0.0       3606.2

In [9]:
# Accuracy grid: rows are depth and kind, columns are budgets.
by_depth["accuracy"].unstack("budget")

budget         4000   8000   16000
d kind                            
1 dependent      0.2    1.0    1.0
  independent    0.4    1.0    1.0
2 dependent      0.2    1.0    1.0
  independent    0.5    1.0    1.0
4 dependent      0.1    1.0    1.0
  independent    0.5    1.0    1.0
6 dependent      0.3    1.0    1.0
  independent    0.5    1.0    1.0

## Reading the results

- Pick the middle budget where the single agent usually finishes (`length_limit` low) but is not near 100% at the larger depths.
- Keep depths whose accuracy at that budget sits roughly between 20% and 80% for at least one kind. Drop depths that are saturated or near zero at every budget.
- If even `d = 6` is near 100%, raise `EQUATIONS` and add larger depths. If `d = 1` is already low, the task is too hard at these budgets.

With 10 pairs per depth, each problem is 10 percentage points within a kind, so treat this as a coarse calibration.